# **Joint single-cell DNA-RNA Data Processing**

## Description of Notebook

Fill in later.

# 0. Import Packages, Install Dependencies

In [40]:
# Import Initial Packages
import os
import sys
import subprocess
import glob

import pandas as pd
import numpy as np
print(pd.__version__)   # should be ≥2.3 and <3.0
print(np.__version__)   # should be ≥1.26 and <2.1

2.2.2
2.0.2


In [ ]:
# Install Dependencies
!pip install snapatac2-scooby

# Installing collected packages: texttable, cykhash, rustworkx, logistro, igraph, array-api-compat, pytest-timeout, pyfaidx, 
# # choreographer, kaleido, hmmlearn, anndata, macs3, snapatac2-scooby


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.8/713.8 kB 11.3 MB/s eta 0:00:00a 0:00:01


: 

In [ ]:
!pip install -q alphagenome scvi-tools scanpy anndata pysam pyranges scrublet gdown

In [ ]:
!pip install --upgrade google-cloud-bigquery



E: Unable to locate package entrez-direct
/bin/bash: line 1: fasterq-dump: command not found
/bin/bash: line 1: esearch: command not found


In [3]:
%%bash
set -e

# Download and install Entrez Direct into $HOME/edirect
sh -c "$(curl -fsSL https://ftp.ncbi.nlm.nih.gov/entrez/entrezdirect/install-edirect.sh)"

# Show where it was installed
# ls -l $HOME/edirect


Entrez Direct has been successfully downloaded and installed.

In order to complete the configuration process, please execute the following:

  echo "export PATH=/root/edirect:\${PATH}" >> ${HOME}/.bashrc

or manually edit the PATH variable assignment in your .bashrc file.

Would you like to do that automatically now? [y/N]
Holding off, then.

To activate EDirect for this terminal session, please execute the following:

export PATH=${HOME}/edirect:${PATH}



In [4]:
# Add Entrez Direct to PATH for Python Kernel
os.environ["PATH"] = os.path.join(os.environ["HOME"], "edirect") + ":" + os.environ["PATH"]
print(os.environ["PATH"])

/root/edirect:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin


In [5]:
# Install NCBI Entrez Direct utilities
#!apt-get install -y -qq entrez-direct 

# Verify installation
#!fasterq-dump --version
print(os.environ["PATH"])

!esearch -help | head -5

/root/edirect:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
esearch 25.3

Query Specification

  -db            Database name


In [6]:
%%bash
# Directly install SRA Toolkit from NCBI Server
set -e

cd $HOME
# Get latest Linux 64‑bit build (adjust URL if you need a different platform)
curl -L -o sratoolkit.tar.gz \
  https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz

tar -xzf sratoolkit.tar.gz
ls

edirect
sratoolkit.3.4.1-ubuntu64
sratoolkit.tar.gz


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 85.0M  100 85.0M    0     0  31.1M      0  0:00:02  0:00:02 --:--:-- 31.1M


In [7]:
# Add SRA Toolkit to PATH variable

home = os.environ["HOME"]
# Find the extracted sratoolkit directory
toolkits = glob.glob(os.path.join(home, "sratoolkit.*-ubuntu64"))
assert toolkits, "No sratoolkit directory found"
sra_dir = os.path.join(toolkits[0], "bin")

os.environ["PATH"] = sra_dir + ":" + os.environ["PATH"]
print(os.environ["PATH"])

/root/sratoolkit.3.4.1-ubuntu64/bin:/root/edirect:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin


In [3]:
# Verify installation
!fasterq-dump --version

/bin/bash: line 1: fasterq-dump: command not found


In [ ]:

#import anndata as ad
#import scanpy as sc



complete


In [4]:
# CONNECT TO GOOGLE DRIVE
import google.colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load files from Google Cloud BigQuery (optional)

# Authenticate Session
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

# Set up GCP Project
#project_id = 'bmi-702-project' # Replace with your actual project ID
#from google.cloud import bigquery
#client = bigquery.Client(project=project_id)

Authenticated


In [12]:
print(os.getcwd)

<built-in function getcwd>


# 1. Collect GEO GSE185269 Files

## a. Download GEO Data to Google Drive

In [11]:
# Download processed data (countmatrices) from GEO Project
# Claude suggested code
import subprocess

# Define output directory
output_dir = "/content/drive/MyDrive/MIT HST 506/project_data/GEO_GSE185269_files"
os.makedirs(output_dir, exist_ok=True)

# Download GEO supplementary files directly via FTP
geo_base = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE185nnn/GSE185269/suppl/"

# List available supplementary files
print("Available files:")
!curl -s {geo_base} | grep -oP '(?<=href=")[^"]+\.gz'

# Download all .gz supplementary files to the output directory
!wget -r -nd -np -A "*.gz" -P "{output_dir}" {geo_base}

# Verify download
!ls -lh "{output_dir}"

Available files:
GSE185269_10x_GBM_Gene_UMI_counts_matrix.csv.gz
GSE185269_10x_GBM_annotation.csv.gz
GSE185269_Bulk_benchmark_WGS_NormalizedCounts.txt.gz
GSE185269_Bulk_benchmark_WGS_RawCounts.txt.gz
GSE185269_Smartseq2_HCT116_RNA_gene_counts_matrix.csv.gz
GSE185269_Smartseq2_PBMC_RNA_gene_counts_matrix.csv.gz
GSE185269_scONE_HCT116_nuclei_gene_counts_expr.csv.gz
GSE185269_scONE_HCT116_whole_gene_counts_expr.csv.gz
GSE185269_scONEseq_Cellline_RNA_gene_counts_matrix.csv.gz
GSE185269_scONEseq_Cellline_scWGS_normalized_counts_matrix.txt.gz
GSE185269_scONEseq_GBM_RNA_gene_counts_matrix.csv.gz
GSE185269_scONEseq_GBM_sample_annotation.csv.gz
GSE185269_scONEseq_GBM_scWGS_mBAF_matrix.txt.gz
GSE185269_scONEseq_GBM_scWGS_normalized_counts_matrix.txt.gz
GSE185269_scONEseq_PBMC_RNA_gene_counts_matrix.csv.gz
--2026-04-20 14:47:25--  https://ftp.ncbi.nlm.nih.gov/geo/series/GSE185nnn/GSE185269/suppl/
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.13, 130.14.250.31, 2607:f220:41e:

## b. Read in GEO Data Files

In [29]:
# Folder and file paths
folder_path = '/content/drive/MyDrive/MIT HST 506/project_data'
GEO_metadata_path_1 = '/content/drive/MyDrive/MIT HST 506/project_data/GEO_GSE185269_files/GSE185269-GPL18573_series_matrix_edited.txt'
GEO_metadata_path_2 = '/content/drive/MyDrive/MIT HST 506/project_data/GEO_GSE185269_files/GSE185269-GPL28038_series_matrix_edited.txt'

GEO_series_matrix_1 = (pd.read_table(GEO_metadata_path_1)).transpose()
GEO_series_matrix_2 = (pd.read_table(GEO_metadata_path_2)).transpose()

# Make first row the column names
GEO_series_matrix_1.columns = GEO_series_matrix_1.iloc[0]
GEO_series_matrix_1.drop(GEO_series_matrix_1.index[0], inplace=True)

GEO_series_matrix_2.columns = GEO_series_matrix_2.iloc[0]
GEO_series_matrix_2.drop(GEO_series_matrix_2.index[0], inplace=True)

In [31]:
(GEO_series_matrix_2).head()

Sample_title,Sample_geo_accession,Sample_status,Sample_submission_date,Sample_last_update_date,Sample_type,Sample_channel_count,Sample_source_name_ch1,Sample_organism_ch1,Sample_characteristics_ch1,Sample_characteristics_ch1,...,Sample_instrument_model,Sample_library_selection,Sample_library_source,Sample_library_strategy,Sample_relation,Sample_relation,Sample_supplementary_file_1,series_matrix_table_begin,ID_REF,series_matrix_table_end
"HCT116, live whole cell, cell 001",GSM6682926,Public on Dec 31 2022,Oct 26 2022,Dec 31 2022,SRA,1,Colon,Homo sapiens,tissue: Colon,cell line: HCT-116,...,DNBSEQ-G400,other,transcriptomic,OTHER,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,NONE,NaN,GSM6682926,NaN
"HCT116, live whole cell, cell 002",GSM6682928,Public on Dec 31 2022,Oct 26 2022,Dec 31 2022,SRA,1,Colon,Homo sapiens,tissue: Colon,cell line: HCT-116,...,DNBSEQ-G400,other,transcriptomic,OTHER,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,NONE,NaN,GSM6682928,NaN
"HCT116, live whole cell, cell 003",GSM6682929,Public on Dec 31 2022,Oct 26 2022,Dec 31 2022,SRA,1,Colon,Homo sapiens,tissue: Colon,cell line: HCT-116,...,DNBSEQ-G400,other,transcriptomic,OTHER,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,NONE,NaN,GSM6682929,NaN
"HCT116, live whole cell, cell 004",GSM6682930,Public on Dec 31 2022,Oct 26 2022,Dec 31 2022,SRA,1,Colon,Homo sapiens,tissue: Colon,cell line: HCT-116,...,DNBSEQ-G400,other,transcriptomic,OTHER,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,NONE,NaN,GSM6682930,NaN
"HCT116, live whole cell, cell 005",GSM6682931,Public on Dec 31 2022,Oct 26 2022,Dec 31 2022,SRA,1,Colon,Homo sapiens,tissue: Colon,cell line: HCT-116,...,DNBSEQ-G400,other,transcriptomic,OTHER,BioSample: https://www.ncbi.nlm.nih.gov/biosam...,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,NONE,NaN,GSM6682931,NaN


# 2. Collect the Run Metadata for SRA SRP339960

In [8]:
items = os.listdir('/content/drive/MyDrive/MIT HST 506/project_data/SRA_SRP339960_files')
print(items)

['.DS_Store', 'SRP339960_runinfo_original.csv', 'SRP339960_runinfo.csv']


In [ ]:
# Fetch the Run Metadata for SRP339960 study (method using Entrez commands)

# !esearch -db sra -query SRP339960 | efetch -format runinfo > /content/drive/MyDrive/"MIT HST 506"/project_data/SRA_SRP339960_files/SRP339960_runinfo.csv

In [9]:
# Folder and file paths
folder_path = '/content/drive/MyDrive/MIT HST 506/project_data'
SRA_metadata_path = '/content/drive/MyDrive/MIT HST 506/project_data/SRA_SRP339960_files/SRP339960_runinfo.csv'

SRA_metadata = pd.read_csv(SRA_metadata_path)

In [10]:
(SRA_metadata).head()

,Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,...,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
0,SRR16195160,2022-12-31 00:17:52,2021-10-04 17:05:27,442928,63422994,442928,143,24,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,7F5445FC0D80D205A571A73A84A9813E,4624C3F8688710092CB2E423492A75BB
1,SRR16195159,2022-12-31 00:17:52,2021-10-04 17:05:27,289373,41434581,289373,143,16,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,3068B3F5E9C0C2F9A3A972C5C4889108,D629EB7BAA322A8F5F0350F4B06C4896
2,SRR16195158,2022-12-31 00:17:52,2021-10-04 17:05:28,330976,47390825,330976,143,18,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,60F618611CE76F02E8C973729976403F,55E10B7273275C5C5823327EE077CD98
3,SRR16195157,2022-12-31 00:17:52,2021-10-04 17:05:44,1433760,205376026,1433760,143,82,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D152661CFEAA7B2875DAC9D5CF5E706E,27458B2217D5EA134A6F093CC0330A1F
4,SRR16195156,2022-12-31 00:17:52,2021-10-04 17:05:46,1249934,179029075,1249934,143,72,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D8755140C9235842B99A1295DAEE5CAC,9C8F0FA05CE1662288524F1170EB7260


In [12]:
# Load and inspect

runinfo = pd.read_csv("/content/drive/MyDrive/MIT HST 506/project_data/SRA_SRP339960_files/SRP339960_runinfo.csv")
print(runinfo.shape)
print(runinfo.columns.tolist())
runinfo.head(10)

(621, 47)
['Run', 'ReleaseDate', 'LoadDate', 'spots', 'bases', 'spots_with_mates', 'avgLength', 'size_MB', 'AssemblyName', 'download_path', 'Experiment', 'LibraryName', 'LibraryStrategy', 'LibrarySelection', 'LibrarySource', 'LibraryLayout', 'InsertSize', 'InsertDev', 'Platform', 'Model', 'SRAStudy', 'BioProject', 'Study_Pubmed_id', 'ProjectID', 'Sample', 'BioSample', 'SampleType', 'TaxID', 'ScientificName', 'SampleName', 'g1k_pop_code', 'source', 'g1k_analysis_group', 'Subject_ID', 'Sex', 'Disease', 'Tumor', 'Affection_Status', 'Analyte_Type', 'Histological_Type', 'Body_Site', 'CenterName', 'Submission', 'dbgap_study_accession', 'Consent', 'RunHash', 'ReadHash']


,Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,...,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
0,SRR16195160,2022-12-31 00:17:52,2021-10-04 17:05:27,442928,63422994,442928,143,24,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,7F5445FC0D80D205A571A73A84A9813E,4624C3F8688710092CB2E423492A75BB
1,SRR16195159,2022-12-31 00:17:52,2021-10-04 17:05:27,289373,41434581,289373,143,16,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,3068B3F5E9C0C2F9A3A972C5C4889108,D629EB7BAA322A8F5F0350F4B06C4896
2,SRR16195158,2022-12-31 00:17:52,2021-10-04 17:05:28,330976,47390825,330976,143,18,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,60F618611CE76F02E8C973729976403F,55E10B7273275C5C5823327EE077CD98
3,SRR16195157,2022-12-31 00:17:52,2021-10-04 17:05:44,1433760,205376026,1433760,143,82,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D152661CFEAA7B2875DAC9D5CF5E706E,27458B2217D5EA134A6F093CC0330A1F
4,SRR16195156,2022-12-31 00:17:52,2021-10-04 17:05:46,1249934,179029075,1249934,143,72,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D8755140C9235842B99A1295DAEE5CAC,9C8F0FA05CE1662288524F1170EB7260
5,SRR16195155,2022-12-31 00:17:52,2021-10-04 17:05:48,1249873,179028128,1249873,143,72,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,A030D8031AD0F81019F804F0DDBFC276,48036FDD9DC94429BC9BDA44B2A4ED3C
6,SRR16195154,2022-12-31 00:17:52,2021-10-04 17:05:45,1229207,176079810,1229207,143,70,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,E475C90208DE4129E051CF73D9EAFA98,2A7F69DBCE6B811F488FD7450B0D53C6
7,SRR16195153,2022-12-31 00:17:52,2021-10-04 17:05:41,1024753,146792703,1024753,143,59,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,67DA2996604182344AD26D7EE1AF14B5,5FAA152A579A111C612A94EFD9F3606A
8,SRR16195152,2022-12-31 00:17:52,2021-10-04 17:05:36,1015576,145446512,1015576,143,58,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,CF0AEF250FF730207A569BF366135B43,0C2F1867F5C907379937020DB94ABDFA
9,SRR16195151,2022-12-31 00:17:52,2021-10-04 17:05:36,713817,102235095,713817,143,41,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,263EBD86D10A955037FE225C970E5AB5,E72940FD9479ABFBDC9A96D6E4B8E0F2


# 3. Distinguish DNA and RNA observations, cell origin

## Separate scDNA and scRNA Runs

In [23]:
# Separate WGS (DNA) and RNA-seq runs
dna_runs = runinfo[runinfo['LibraryStrategy'] == 'OTHER']['Run'].tolist()
rna_runs = runinfo[runinfo['LibraryStrategy'] == 'RNA-Seq']['Run'].tolist()

print(f"DNA (WGS) runs: {len(dna_runs)}")
print(f"RNA-seq runs:   {len(rna_runs)}")
print("\nDNA runs:", dna_runs[:5])
print("RNA runs:", rna_runs[:5])

DNA (WGS) runs: 525
RNA-seq runs:   96

DNA runs: ['SRR16195157', 'SRR16195156', 'SRR16195155', 'SRR16195154', 'SRR16195153']
RNA runs: ['SRR16195160', 'SRR16195159', 'SRR16195158', 'SRR16195109', 'SRR16195108']


In [ ]:
# Sample Name, Library Name, BioSample
print(f"Number of unique Runs: {len((runinfo['Run']).unique())}") #621
print(f"Number of unique Sample Names: {len((runinfo['SampleName']).unique())}") #621
print(f"Number of unique Library Names: {len((runinfo['LibraryName']).unique())}") #621
print(f"Number of unique BioSamples: {len((runinfo['BioSample']).unique())}") #621
print(f"Number of unique Experiments: {len((runinfo['Experiment']).unique())}") #621
print(f"Number of unique Samples: {len((runinfo['Sample']).unique())}") #621


Number of unique Runs: 621
Number of unique Sample Names: 621
Number of unique Library Names: 621
Number of unique BioSamples: 621
Number of unique Experiments: 621
Number of unique Samples: 621


In [13]:
(runinfo[["Run", "SampleName", "LibraryName", "BioSample", "Experiment", "Sample"]]).head(10)

,Run,SampleName,LibraryName,BioSample,Experiment,Sample
0,SRR16195160,GSM5609425,GSM5609425,SAMN22044904,SRX12479394,SRS10442945
1,SRR16195159,GSM5609426,GSM5609426,SAMN22044903,SRX12479395,SRS10442946
2,SRR16195158,GSM5609427,GSM5609427,SAMN22044902,SRX12479396,SRS10442947
3,SRR16195157,GSM5609620,GSM5609620,SAMN22044709,SRX12479397,SRS10442948
4,SRR16195156,GSM5609621,GSM5609621,SAMN22044708,SRX12479398,SRS10442949
5,SRR16195155,GSM5609622,GSM5609622,SAMN22044707,SRX12479399,SRS10442950
6,SRR16195154,GSM5609623,GSM5609623,SAMN22044706,SRX12479400,SRS10442951
7,SRR16195153,GSM5609624,GSM5609624,SAMN22044705,SRX12479401,SRS10442952
8,SRR16195152,GSM5609625,GSM5609625,SAMN22044704,SRX12479402,SRS10442953
9,SRR16195151,GSM5609626,GSM5609626,SAMN22044703,SRX12479403,SRS10442954


## Preprocess GEO Series Matrix 1 (GPL18573)

In [ ]:
GEO_series_matrix_1_processed = GEO_series_matrix_1.copy() #431 x 48
# IDEA: join SRA metadata and GEO series matrix 1 on their unique SampleName/Sample_geo_accession.

# Rename duplicate columns
from collections import Counter

cols = list(GEO_series_matrix_1_processed.columns)
seen = Counter()
new_cols = []

# keep the first column name exactly
new_cols.append(cols[0])
seen[cols[0]] += 1

# process the rest, de‑duplicating as needed
for c in cols[1:]:
    seen[c] += 1
    if seen[c] == 1:
        new_cols.append(c)
    else:
        new_cols.append(f"{c}_{seen[c]}")

GEO_series_matrix_1_processed.columns = new_cols

GEO_series_matrix_1_processed.index.name = 'Sample_title'
GEO_series_matrix_1_processed = GEO_series_matrix_1_processed.reset_index()

In [73]:
# GEO series matrix 1: 431 rows x 49 columns

# Variables to keep: 
# Sample_title, Sample_geo_accession, Sample_source_name_ch1, Sample_library_selection, Sample_library_source
# Sample_library_strategy, Sample_characteristics_ch1_2, Sample_description, Sample_description_2, Sample_platform_id
# Sample_instrument_model, 

# Variables to remove: 
# Sample_status, Sample_submission_date, Sample_last_update, Sample_type, Sample_channel_count
# Sample_organism_ch1, Sample_characteristics_ch1, Sample_instrument_model, Sample_relation, Sample_relation_2
# Sample_supplementary_file_1, series_matrix_table_begin, ID_REF, series_matrix_table_end
# Sample_characteristics_ch1_2, Sample_growth_protocol_ch1, Sample_molecule_ch1, Sample_extract_protocol_ch1, 
# Sample_extract_protocol_ch1_2, Sample_taxid_ch1
# 'Sample_data_processing', 'Sample_data_processing_2', 'Sample_data_processing_3', 'Sample_data_processing_4', 'Sample_data_processing_5', 
# 'Sample_data_processing_6','Sample_data_processing_7', 'Sample_data_processing_8', 'Sample_data_processing_9', 'Sample_data_processing_10',
# 'Sample_contact_name', 'Sample_contact_email', 'Sample_contact_laboratory', 'Sample_contact_department', 'Sample_contact_institute', 
# 'Sample_contact_address','Sample_contact_city', 'Sample_contact_zip/postal_code', 'Sample_contact_country', 'Sample_data_row_count',

vars_to_keep = [
"Sample_title", "Sample_geo_accession", "Sample_source_name_ch1", "Sample_library_selection", 
"Sample_library_source", "Sample_library_strategy", "Sample_characteristics_ch1_2", "Sample_description",
 "Sample_description_2", "Sample_platform_id", "Sample_instrument_model"
]

GEO_series_matrix_1_processed = GEO_series_matrix_1_processed[vars_to_keep]

In [75]:
# Fix sample_description column
GEO_series_matrix_1_processed["Sample_description"] = GEO_series_matrix_1_processed["Sample_description"].replace("RNA+DNA", np.nan)

# New column with the cell info from whichever column is non‑NaN
GEO_series_matrix_1_processed["Sample_description"] = GEO_series_matrix_1_processed["Sample_description"].fillna(
    GEO_series_matrix_1_processed["Sample_description_2"]
)

# Remove unnecessary columns
GEO_series_matrix_1_processed = GEO_series_matrix_1_processed.drop(columns=["Sample_description_2"])



In [108]:
(GEO_series_matrix_1_processed["Sample_characteristics_ch1_2"]) #431 unique values

,Sample_characteristics_ch1_2
0,method: Smart-seq2
1,method: Smart-seq2
2,method: Smart-seq2
3,method: Smart-seq2
4,method: Smart-seq2
...,...
426,method: scONE-seq
427,method: scONE-seq
428,method: scONE-seq
429,method: scONE-seq


In [ ]:
GEO_series_matrix_1_processed.to_csv("/content/drive/MyDrive/MIT HST 506/project_data/GSE185269-GPL18573_series_matrix_processed.csv")

## Preprocess GEO Series Matrix 2 (GPL18573)

In [ ]:
# GEO series matrix 2: 47 columns
GEO_series_matrix_2_processed = GEO_series_matrix_2.copy() 

# Rename duplicate columns
from collections import Counter

cols = list(GEO_series_matrix_2_processed.columns)
seen = Counter()
new_cols = []

# keep the first column name exactly
new_cols.append(cols[0])
seen[cols[0]] += 1

# process the rest, de‑duplicating as needed
for c in cols[1:]:
    seen[c] += 1
    if seen[c] == 1:
        new_cols.append(c)
    else:
        new_cols.append(f"{c}_{seen[c]}")

GEO_series_matrix_2_processed.columns = new_cols

GEO_series_matrix_2_processed.index.name = 'Sample_title'
GEO_series_matrix_2_processed = GEO_series_matrix_2_processed.reset_index()

In [ ]:
# GEO series matrix 2: #190 rows x 47 columns

# Variables to keep: 
# Sample_title, Sample_geo_accession, Sample_characteristics_ch1_2, Sample_description, Sample_platform_id
# Sample_library_selection, Sample_library_source, Sample_library_strategy, Sample_instrument_model


# Variables to remove: 
# Sample_status, Sample_submission_date, Sample_last_update, Sample_type, Sample_channel_count
# Sample_organism_ch1, Sample_characteristics_ch1, Sample_instrument_model, Sample_relation, Sample_relation_2
# Sample_supplementary_file_1, series_matrix_table_begin, ID_REF, series_matrix_table_end
# Sample_growth_protocol_ch1, Sample_molecule_ch1, Sample_extract_protocol_ch1, 
# Sample_extract_protocol_ch1_2, Sample_taxid_ch1, Sample_molecule_ch1, Sample_source_name_ch1
# 'Sample_data_processing', 'Sample_data_processing_2', 'Sample_data_processing_3', 'Sample_data_processing_4', 'Sample_data_processing_5', 
# 'Sample_data_processing_6','Sample_data_processing_7', 'Sample_data_processing_8', 'Sample_data_processing_9', 'Sample_data_processing_10',
# 'Sample_contact_name', 'Sample_contact_email', 'Sample_contact_laboratory', 'Sample_contact_department', 'Sample_contact_institute', 
# 'Sample_contact_address','Sample_contact_city', 'Sample_contact_zip/postal_code', 'Sample_contact_country', 'Sample_data_row_count',


vars_to_keep_2 = [
"Sample_title", "Sample_geo_accession", "Sample_characteristics_ch1_2", "Sample_description", "Sample_platform_id",
"Sample_library_selection", "Sample_library_source", "Sample_library_strategy", "Sample_instrument_model"
]

GEO_series_matrix_2_processed = GEO_series_matrix_2_processed[vars_to_keep_2]

GEO_series_matrix_2_processed["Sample_description"] = GEO_series_matrix_2_processed["Sample_title"]

In [ ]:
GEO_series_matrix_2_processed.head(5)



,Sample_title,Sample_geo_accession,Sample_characteristics_ch1_2,Sample_description,Sample_platform_id,Sample_library_selection,Sample_library_source,Sample_library_strategy,Sample_instrument_model
0,"HCT116, live whole cell, cell 001",GSM6682926,cell line: HCT-116,"HCT116, live whole cell, cell 001",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
1,"HCT116, live whole cell, cell 002",GSM6682928,cell line: HCT-116,"HCT116, live whole cell, cell 002",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
2,"HCT116, live whole cell, cell 003",GSM6682929,cell line: HCT-116,"HCT116, live whole cell, cell 003",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
3,"HCT116, live whole cell, cell 004",GSM6682930,cell line: HCT-116,"HCT116, live whole cell, cell 004",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
4,"HCT116, live whole cell, cell 005",GSM6682931,cell line: HCT-116,"HCT116, live whole cell, cell 005",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
5,"HCT116, live whole cell, cell 006",GSM6682932,cell line: HCT-116,"HCT116, live whole cell, cell 006",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
6,"HCT116, live whole cell, cell 007",GSM6682933,cell line: HCT-116,"HCT116, live whole cell, cell 007",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
7,"HCT116, live whole cell, cell 008",GSM6682934,cell line: HCT-116,"HCT116, live whole cell, cell 008",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
8,"HCT116, live whole cell, cell 009",GSM6682935,cell line: HCT-116,"HCT116, live whole cell, cell 009",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400
9,"HCT116, live whole cell, cell 010",GSM6682936,cell line: HCT-116,"HCT116, live whole cell, cell 010",GPL28038,other,transcriptomic,OTHER,DNBSEQ-G400


In [105]:
GEO_series_matrix_2_processed.to_csv("/content/drive/MyDrive/MIT HST 506/project_data/GSE185269-GPL28038_series_matrix_processed.csv")

## Concatenate GEO Series Matrices

In [ ]:
#GEO_series_matrix_1_processed.head()

# Rename Sample_source_name_ch1 to Sample_cell_line
GEO_series_matrix_1_processed.rename(columns={'Sample_source_name_ch1': 'Sample_cell_line'}, inplace=True)

# Rename Sample_description to Sample_cell_description
GEO_series_matrix_1_processed.rename(columns={'Sample_description': 'Sample_cell_description'}, inplace=True)

# Remove Sample_characteristics_ch1_2
GEO_series_matrix_1_processed = GEO_series_matrix_1_processed.drop(columns=["Sample_characteristics_ch1_2"])

,Sample_title,Sample_geo_accession,Sample_cell_line,Sample_library_selection,Sample_library_source,Sample_library_strategy,Sample_cell_description,Sample_platform_id,Sample_instrument_model
0,Smart-HCT-01,GSM5609425,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 001,GPL18573,Illumina NextSeq 500
1,Smart-HCT-02,GSM5609426,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 002,GPL18573,Illumina NextSeq 500
2,Smart-HCT-03,GSM5609427,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 003,GPL18573,Illumina NextSeq 500
3,Smart-HCT-04,GSM5609428,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 004,GPL18573,Illumina NextSeq 500
4,Smart-HCT-05,GSM5609429,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 005,GPL18573,Illumina NextSeq 500
...,...,...,...,...,...,...,...,...,...
426,NPC43 scONEseq 166,GSM5609851,NPC43 cell line,other,transcriptomic,OTHER,NPC single cell 166,GPL18573,Illumina NextSeq 500
427,NPC43 scONEseq 167,GSM5609852,NPC43 cell line,other,transcriptomic,OTHER,NPC single cell 167,GPL18573,Illumina NextSeq 500
428,NPC43 scONEseq 168,GSM5609853,NPC43 cell line,other,transcriptomic,OTHER,NPC single cell 168,GPL18573,Illumina NextSeq 500
429,NPC43 scONEseq 169,GSM5609854,NPC43 cell line,other,transcriptomic,OTHER,NPC single cell 169,GPL18573,Illumina NextSeq 500


In [ ]:
# Rename Sample_characteristics_ch1_2 to Sample_cell_line
GEO_series_matrix_2_processed.rename(columns={'Sample_characteristics_ch1_2': 'Sample_cell_line'}, inplace=True)

# Rename Sample_description to Sample_cell_description
GEO_series_matrix_2_processed.rename(columns={'Sample_description': 'Sample_cell_description'}, inplace=True)


/tmp/ipykernel_1516/2529234801.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  GEO_series_matrix_2_processed.rename(columns={'Sample_characteristics_ch1_2': 'Sample_cell_line'}, inplace=True)
/tmp/ipykernel_1516/2529234801.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  GEO_series_matrix_2_processed.rename(columns={'Sample_description': 'Sample_cell_description'}, inplace=True)


In [ ]:
GEO_series_matrix_combined = pd.concat([GEO_series_matrix_1_processed, GEO_series_matrix_2_processed], ignore_index = True)
GEO_series_matrix_combined.to_csv("/content/drive/MyDrive/MIT HST 506/project_data/GEO_series_matrix_combined.csv")


In [123]:
GEO_series_matrix_combined.head()

,Sample_title,Sample_geo_accession,Sample_cell_line,Sample_library_selection,Sample_library_source,Sample_library_strategy,Sample_cell_description,Sample_platform_id,Sample_instrument_model
0,Smart-HCT-01,GSM5609425,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 001,GPL18573,Illumina NextSeq 500
1,Smart-HCT-02,GSM5609426,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 002,GPL18573,Illumina NextSeq 500
2,Smart-HCT-03,GSM5609427,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 003,GPL18573,Illumina NextSeq 500
3,Smart-HCT-04,GSM5609428,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 004,GPL18573,Illumina NextSeq 500
4,Smart-HCT-05,GSM5609429,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 005,GPL18573,Illumina NextSeq 500


## Join GEO series metadata with SRA Metadata

In [ ]:
runinfo.head()
# New df name: full_metadata
# Join runinfo and GEO_series_matrix_combined: runinfo["SampleName"] and GEO_series_matrix_combined["Sample_geo_accession"]

,Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,...,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
0,SRR16195160,2022-12-31 00:17:52,2021-10-04 17:05:27,442928,63422994,442928,143,24,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,7F5445FC0D80D205A571A73A84A9813E,4624C3F8688710092CB2E423492A75BB
1,SRR16195159,2022-12-31 00:17:52,2021-10-04 17:05:27,289373,41434581,289373,143,16,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,3068B3F5E9C0C2F9A3A972C5C4889108,D629EB7BAA322A8F5F0350F4B06C4896
2,SRR16195158,2022-12-31 00:17:52,2021-10-04 17:05:28,330976,47390825,330976,143,18,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,60F618611CE76F02E8C973729976403F,55E10B7273275C5C5823327EE077CD98
3,SRR16195157,2022-12-31 00:17:52,2021-10-04 17:05:44,1433760,205376026,1433760,143,82,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D152661CFEAA7B2875DAC9D5CF5E706E,27458B2217D5EA134A6F093CC0330A1F
4,SRR16195156,2022-12-31 00:17:52,2021-10-04 17:05:46,1249934,179029075,1249934,143,72,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,NaN,NaN,NaN,NaN,"ANGELA WU LAB, LIFS, HKUST",SRA1305581,NaN,public,D8755140C9235842B99A1295DAEE5CAC,9C8F0FA05CE1662288524F1170EB7260


In [125]:
# Use left merge to join matrices
#GEO_series_matrix_combined["Sample_geo_accession"]

geo_meta = GEO_series_matrix_combined.rename(
    columns={"Sample_geo_accession": "SampleName"}
)

full_metadata = runinfo.merge(
geo_meta, 
on = "SampleName",
how = "left"
)


In [127]:
full_metadata.to_csv("/content/drive/MyDrive/MIT HST 506/project_data/full_metadata.csv")

In [129]:
full_metadata.head() # 621 rows × 55 columns

,Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,...,RunHash,ReadHash,Sample_title,Sample_cell_line,Sample_library_selection,Sample_library_source,Sample_library_strategy,Sample_cell_description,Sample_platform_id,Sample_instrument_model
0,SRR16195160,2022-12-31 00:17:52,2021-10-04 17:05:27,442928,63422994,442928,143,24,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,7F5445FC0D80D205A571A73A84A9813E,4624C3F8688710092CB2E423492A75BB,Smart-HCT-01,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 001,GPL18573,Illumina NextSeq 500
1,SRR16195159,2022-12-31 00:17:52,2021-10-04 17:05:27,289373,41434581,289373,143,16,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,3068B3F5E9C0C2F9A3A972C5C4889108,D629EB7BAA322A8F5F0350F4B06C4896,Smart-HCT-02,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 002,GPL18573,Illumina NextSeq 500
2,SRR16195158,2022-12-31 00:17:52,2021-10-04 17:05:28,330976,47390825,330976,143,18,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,60F618611CE76F02E8C973729976403F,55E10B7273275C5C5823327EE077CD98,Smart-HCT-03,HCT116 cell line,cDNA,transcriptomic,RNA-Seq,HCT116 single cell 003,GPL18573,Illumina NextSeq 500
3,SRR16195157,2022-12-31 00:17:52,2021-10-04 17:05:44,1433760,205376026,1433760,143,82,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,D152661CFEAA7B2875DAC9D5CF5E706E,27458B2217D5EA134A6F093CC0330A1F,HUVEC+H9 scONEseq 002,HUVEC+H9 cell line co-culture,other,transcriptomic,OTHER,Normal single cell 002,GPL18573,Illumina NextSeq 500
4,SRR16195156,2022-12-31 00:17:52,2021-10-04 17:05:46,1249934,179029075,1249934,143,72,NaN,https://sra-downloadb.be-md.ncbi.nlm.nih.gov/s...,...,D8755140C9235842B99A1295DAEE5CAC,9C8F0FA05CE1662288524F1170EB7260,HUVEC+H9 scONEseq 003,HUVEC+H9 cell line co-culture,other,transcriptomic,OTHER,Normal single cell 003,GPL18573,Illumina NextSeq 500


In [ ]:
metadata_subset = full_metadata[
    "Run", "LibraryName", "LibraryStrategy", "LibrarySelection", "SampleName", "Sample_title", "Sample_cell_line",
    "Sample_library_selection", "Sample_library_sourcce", "Sample_library_strategy", "Simple_cell_description",
    "Sample_platform_id", "Sample_instrument_model"
    ]

# 4. Download FASTQ Files

## a. Download a small subset first to test

In [ ]:
# Define base path on Google Drive
base_path = "/content/drive/MyDrive/MIT HST 506/project_data/FASTQ_files"

os.makedirs(f"{base_path}/dna", exist_ok=True)
os.makedirs(f"{base_path}/rna", exist_ok=True)

# Test with first DNA run
test_dna = dna_runs[0]
!fasterq-dump {test_dna} --split-files --outdir "{base_path}/dna" --threads 4
!gzip "{base_path}/dna/{test_dna}"*.fastq

# Test with first RNA run
test_rna = rna_runs[0]
!fasterq-dump {test_rna} --split-files --outdir "{base_path}/rna" --threads 4
!gzip "{base_path}/rna/{test_rna}"*.fastq

!ls -lh "{base_path}/dna/"
!ls -lh "{base_path}/rna/"

spots read      : 1,433,760
reads read      : 2,867,520
reads written   : 2,867,520
spots read      : 442,928
reads read      : 885,856
reads written   : 885,856
total 127M
-rw------- 1 root root 64M Apr 20 03:06 SRR16195157_1.fastq.gz
-rw------- 1 root root 63M Apr 20 03:06 SRR16195157_2.fastq.gz
total 40M
-rw------- 1 root root 20M Apr 20 03:07 SRR16195160_1.fastq.gz
-rw------- 1 root root 20M Apr 20 03:07 SRR16195160_2.fastq.gz
